# Módulo 02: De RDDs a DataFrames y Esquemas Fuertemente Tipados

## 1. Antipatrones del Enfoque Clásico vs. Estándar Moderno

En la era inicial de Spark (1.x y principios de 2.x), era común procesar datos estructurados recurriendo directamente a la API de bajo nivel de **RDDs** (`sc.parallelize`, `map`, `filter`, `reduceByKey`).

### ¿Por qué los RDDs son un antipatrón para datos tabulares?
* **Sobrecosto de serialización (Py4J / Pickle):** Cada transformación en RDD obliga a la JVM a enviar datos en bruto a un subproceso de Python, serializándolos y deserializándolos fila por fila con Pickle.
* **Invisibilidad para el Optimizador:** La JVM trata las funciones de Python como cajas negras. No puede reordenar operaciones, podar columnas no utilizadas ni empujar predicados hacia el almacenamiento.
* **Sobrecarga del Garbage Collector:** Al crear millones de objetos Python individuales en memoria, el recolector de basura impacta negativamente la latencia.

### La Solución Moderna: DataFrames, Catalyst y Tungsten
* **Optimizador Catalyst:** Compila las transformaciones declarativas a través de cuatro fases (Plan Lógico No Resuelto -> Plan Lógico Resuelto -> Plan Optimizado -> Plan Físico), aplicando reglas como *Predicate Pushdown* y *Projection Pruning*.
* **Motor Tungsten:** Administra la memoria fuera del heap de la JVM (*off-heap*) y almacena las columnas en formato binario compacto, ejecutando código Java compilado en tiempo de ejecución (*Whole-Stage Code Generation*).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)
import pyspark.sql.functions as F

# Inicialización canónica
spark = (
    SparkSession.builder
    .appName("02_DataFrames_y_Esquemas")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("SparkSession lista para procesamiento estructurado.")

SparkSession lista para procesamiento estructurado.


## 2. El Peligro de `inferSchema=True` en Producción

En tutoriales introductorios suele usarse `.option("inferSchema", "true")`. En entornos productivos y de ingeniería de datos, **esto es una mala práctica crítica**:

1. **Lectura doble de I/O:** Para inferir tipos, Spark debe escanear el archivo completo una primera vez, determinar los tipos de cada columna y realizar una segunda pasada para instanciar el DataFrame. En archivos de gigabytes o terabytes, esto duplica el tiempo de lectura.
2. **Fragilidad de tipos:** Un único valor nulo anómalo, un salto de línea o una cadena no prevista puede forzar a Spark a interpretar una columna numérica como `StringType`, rompiendo los pipelines posteriores silenciosamente.
3. **Pérdida del contrato de datos:** Sin un esquema explícito (`StructType`), el código carece de validación formal contra cambios inesperados de esquema en origen (*Schema Drift*).

In [2]:
# 1. Definición explícita del contrato de datos (DDL programático)
deportistas_schema = StructType([
    StructField("deportista_id", IntegerType(), nullable=False),
    StructField("nombre", StringType(), nullable=False),
    StructField("genero", IntegerType(), nullable=True),
    StructField("edad", IntegerType(), nullable=True),
    StructField("altura", DoubleType(), nullable=True),
    StructField("peso", DoubleType(), nullable=True),
    StructField("equipo_id", IntegerType(), nullable=True)
])

# 2. Ingesta determinista en una sola pasada de I/O
df_deportistas = (
    spark.read
    .schema(deportistas_schema)
    .option("header", "true")
    .csv("../data/raw/deportista.csv")
)

df_deportistas.printSchema()
df_deportistas.show(5)

root
 |-- deportista_id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: double (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24| 180.0|80.0|        1|
|            2|            A Lamusi|     1|  23| 170.0|60.0|        2|
|            3|      Gunnar Nielsen|     1|  24| 185.0|82.0|        3|
|            4|Edgar Lindenau Aabye|     1|  34| 182.0|81.0|        4|
|            5|Christine Jacoba ...|     2|  21| 185.0|72.0|        5|
+-------------+--------------------+------+----+------+----+---------+
only showing top 5 rows



## 3. Inspección del Optimizador Catalyst con `.explain()`

Para auditar cómo Spark ejecuta una consulta, utilizamos el método `.explain()`. El parámetro `mode="formatted"` divide el resultado en dos secciones legibles:
1. **Physical Plan:** El grafo de operadores que se ejecutará en hardware.
2. **Detailed Plan Details:** Información sobre *PushedFilters* (filtros aplicados en disco) y *ReadSchema* (columnas leídas).

In [3]:
# Declaración de transformaciones (Evaluación Perezosa)
df_consulta = (
    df_deportistas
    .filter((F.col("edad") >= 25) & (F.col("altura") > 175.0))
    .select("deportista_id", "nombre", "edad", "altura")
)

# Inspección del plan físico optimizado
df_consulta.explain(mode="formatted")

== Physical Plan ==
* Filter (2)
+- Scan csv  (1)


(1) Scan csv 
Output [4]: [deportista_id#0, nombre#1, edad#3, altura#4]
Batched: false
Location: InMemoryFileIndex [file:/c:/Users/Ruben/Desktop/Ciencia de datos/curso_spark/data/raw/deportista.csv]
PushedFilters: [IsNotNull(edad), IsNotNull(altura), GreaterThanOrEqual(edad,25), GreaterThan(altura,175.0)]
ReadSchema: struct<deportista_id:int,nombre:string,edad:int,altura:double>

(2) Filter [codegen id : 1]
Input [4]: [deportista_id#0, nombre#1, edad#3, altura#4]
Condition : (((isnotnull(edad#3) AND isnotnull(altura#4)) AND (edad#3 >= 25)) AND (altura#4 > 175.0))




## 4. Reto Práctico: Esquema Estricto y Filtrado Determinista

### Instrucciones del Ejercicio:
1. Diseña un esquema explícito con `StructType` para el archivo `../data/raw/equipos.csv` con los siguientes campos:
   * `equipo_id` (IntegerType, no nulo)
   * `equipo` (StringType, no nulo)
   * `sigla` (StringType, no nulo)
2. Lee el archivo CSV aplicando dicho esquema explícito (sin inferencia automática).
3. Filtra los equipos cuya sigla sea `"DEN"` o `"USA"`.
4. Ordena el resultado por `equipo_id` de forma ascendente.
5. Extrae la lista de identificadores y verifica la solución con la aserción automática.

In [4]:
# 1. Definición del esquema formal
equipos_schema = StructType([
    StructField("equipo_id", IntegerType(), nullable=False),
    StructField("equipo", StringType(), nullable=False),
    StructField("sigla", StringType(), nullable=False)
])

# 2. Lectura estricta
df_equipos = (
    spark.read
    .schema(equipos_schema)
    .option("header", "true")
    .csv("../data/raw/equipos.csv")
)

# 3. Filtrado y ordenamiento
df_filtrado = (
    df_equipos
    .filter(F.col("sigla").isin("DEN", "USA"))
    .orderBy(F.col("equipo_id").asc())
)

df_filtrado.show()

# 4. Extracción de IDs para validación
ids_obtenidos = [row["equipo_id"] for row in df_filtrado.select("equipo_id").collect()]
print(f"IDs obtenidos: {ids_obtenidos}")

# 5. Validación automática
assert ids_obtenidos == [3, 4, 6], f"Aserción fallida: Se esperaba [3, 4, 6] pero se obtuvo {ids_obtenidos}"
print("¡Aserción aprobada! Cuaderno 02 completado con éxito.")

+---------+--------------+-----+
|equipo_id|        equipo|sigla|
+---------+--------------+-----+
|        3|       Denmark|  DEN|
|        4|Denmark/Sweden|  DEN|
|        6| United States|  USA|
+---------+--------------+-----+

IDs obtenidos: [3, 4, 6]
¡Aserción aprobada! Cuaderno 02 completado con éxito.
